### CÀI THƯ VIỆN VÀ IMPORT DỮ LIỆU

In [2]:
import pandas as pd
import numpy as np

products = pd.read_csv("datasets/cleaned_data/products_clean.csv")

In [3]:
products

,id,sku,name,seller_product_id,seller_id,tracking_info_amplitude,inventory_status,inventory_type,data_version,day_ago_created,...,badges_new_has_video_badge,badges_new_has_authentic_badge,badges_new_has_variant_count_badge,badges_new_has_delivery_badge,badges_v3_has_video_badge,badges_v3_has_authentic_badge,badges_v3_has_variant_count_badge,badges_v3_has_delivery_badge,log_quantity_sold,best_seller
0,54645,6508148796624,Tai nghe chụp tai Logitech H150 - 2 jack 3.5mm...,1095356,915,is_authentic:True; is_freeship_xtra:True; is_h...,available,backorder,3300,4775,...,False,True,True,True,False,False,False,False,4.574711,False
1,54665,5007102619128,Chuột không dây Logitech M185 - Hàng chính hãn...,54673,1,is_authentic:True; is_freeship_xtra:False; is_...,available,backorder,3300,4774,...,True,True,True,True,False,False,False,False,8.832734,True
2,122012,6805279483134,Bộ Bàn Phím Và Chuột Không Dây Logitech MK240 ...,122016,1,is_authentic:True; is_freeship_xtra:False; is_...,available,instock,3300,4081,...,True,True,True,True,False,False,False,False,7.491645,True
3,227386,1446930336854,Pin Sạc Dự Phòng Anker PowerCore 20100mAh - A1271,227388,1,is_authentic:True; is_freeship_xtra:False; is_...,available,backorder,3300,3527,...,False,True,True,True,False,False,False,False,7.344073,True
4,299461,3057399746627,Chuột Không Dây Logitech M331 Silent,49019988,27572,is_authentic:True; is_freeship_xtra:True; is_h...,available,backorder,3300,3337,...,False,True,True,True,False,False,False,False,9.221577,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12192,279105357,9978192483262,Son dưỡng môi không màu chuyển kem mịn LipIce ...,278852860,1,is_authentic:True; is_freeship_xtra:True; is_h...,available,instock,3300,13,...,False,True,True,True,False,False,False,False,6.734592,True
12193,279115900,2907591993358,viên rửa Bát Finish Quantum 64 viên - Quantum ...,136803024,101510,is_authentic:True; is_freeship_xtra:True; is_h...,available,backorder,3300,12,...,False,True,True,True,False,False,False,False,2.397895,False
12194,279116238,3947447272170,Túi 60 viên rửa chén Finish Quantum All in 1 (...,270927116,101510,is_authentic:True; is_freeship_xtra:True; is_h...,available,backorder,3300,12,...,False,True,True,True,False,False,False,False,1.386294,False
12195,279134755,3221073047404,Loa Innoyo InnoSound S1 - Hàng chính hãng,279134756,1,is_authentic:True; is_freeship_xtra:False; is_...,available,instock,3300,7,...,False,True,False,True,False,False,False,False,NaN,NaN


In [ ]:
#1. Thống kê tổng quan theo category
base = products[products["price"].notna()].copy() #Lọc bỏ các sản phẩm có giá không xác định (NaN)

summary_category = (
    base
    .groupby("category")
    .agg(
        product_count=("category", "count"),
        avg_price=("price", "mean"),
        avg_discount_rate=("discount_rate", "mean"),
        avg_rating=("rating_average", "mean"),
        total_reviews=("reviews_count", "sum"),
        total_quantity_sold=("quantity_sold", "sum")
    )
    .round({
        "avg_price": 2,
        "avg_discount_rate": 4,
        "avg_rating": 2
    })
    .sort_values("product_count", ascending=False)
    .reset_index()
)

summary_category

,category,product_count,avg_price,avg_discount_rate,avg_rating,total_reviews,total_quantity_sold
0,home_appliances,600,992091.48,24.9850,3.53,61297,337432.0
1,backpacks_luggage,520,822516.22,5.5865,2.32,4537,23913.0
2,beauty_personal_care,520,237766.77,14.7596,4.07,57207,3627330.0
3,books_tiki,520,126548.77,24.9462,4.29,176059,1677826.0
4,consumer_electronics_home_appliances,520,9990815.71,12.2288,1.37,1291,17384.0
5,digital_devices_accessories,520,374505.49,8.2635,3.32,29801,190672.0
6,home_care_cleaning,520,208491.97,4.4212,3.27,28011,298132.0
7,grocery_daily_essentials,520,211031.64,11.0327,4.12,62826,1535444.0
8,automotive_motorcycles,520,11854152.90,2.0769,2.75,8502,38878.0
9,women_fashion,520,213360.41,4.5846,2.14,4671,20710.0


In [5]:
#2. Phân tích phân bổ GMV (Gross Merchandise Value) proxy theo category (kiểm tra 80/20)
gmv_table = (
    products[products["price"].notna() & products["quantity_sold"].notna()]
    .assign(gmv_proxy=lambda x: x["price"].astype("int64") * x["quantity_sold"]) 
    .groupby("category")["gmv_proxy"]
    .sum()
    .reset_index()
)

total_gmv = gmv_table["gmv_proxy"].sum() 

gmv_table["gmv_pct"] = (gmv_table["gmv_proxy"] / total_gmv * 100).round(2)

gmv_table = gmv_table.sort_values("gmv_proxy", ascending=False)

gmv_table["cumulative_gmv_pct"] = (
    gmv_table["gmv_proxy"].cumsum() / total_gmv * 100
).round(2)

gmv_table = gmv_table.reset_index(drop=True)

gmv_table

,category,gmv_proxy,gmv_pct,cumulative_gmv_pct
0,vouchers_services,1.020592e+12,26.59,26.59
1,toys_kids_baby,7.436349e+11,19.37,45.96
2,grocery_daily_essentials,4.137006e+11,10.78,56.74
3,beauty_personal_care,3.730906e+11,9.72,66.46
4,home_appliances,2.978153e+11,7.76,74.22
5,mobile_phones_tablets,2.548475e+11,6.64,80.86
6,books_tiki,1.890234e+11,4.92,85.78
7,consumer_electronics_home_appliances,1.037689e+11,2.70,88.49
8,computers_laptops_components,8.917492e+10,2.32,90.81
9,automotive_motorcycles,8.406028e+10,2.19,93.00


In [6]:
#3. So sánh mức giảm giá trung bình theo category
discount_category = (
    products
    .groupby("category")
    .agg(
        product_count=("category", "count"),
        avg_discount_rate=("discount_rate", "mean")
    )
)

discount_category = (
    discount_category[discount_category["product_count"] >= 30]
    .round({"avg_discount_rate": 4})
    .sort_values("avg_discount_rate", ascending=False)
)
discount_category = discount_category.reset_index()
discount_category

,category,product_count,avg_discount_rate
0,home_appliances,600,24.9850
1,books_tiki,520,24.9462
2,home_lifestyle,520,20.8519
3,beauty_personal_care,520,14.7596
4,sports_outdoors,520,14.0519
5,consumer_electronics_home_appliances,520,12.2288
6,toys_kids_baby,520,11.1212
7,grocery_daily_essentials,520,11.0327
8,mobile_phones_tablets,225,8.4489
9,digital_devices_accessories,520,8.2635


In [7]:
#4. So sánh mức giảm giá trung bình theo brand
pd.set_option('display.max_rows', None)

discount_brand = (
    products[products["brand_name"].notna()]
    .groupby("brand_name")
    .agg(
        product_count=("brand_name", "count"),
        avg_discount_rate=("discount_rate", "mean")
    )
)

discount_brand = (
    discount_brand[discount_brand["product_count"] >= 20]
    .round({"avg_discount_rate": 4})
    .sort_values("avg_discount_rate", ascending=False)
)

discount_brand = discount_brand.reset_index()
discount_brand

,brand_name,product_count,avg_discount_rate
0,Elmich,141,60.9929
1,TP Tiến Phát SMART,36,47.8611
2,Deli,44,41.3409
3,PaKaSa,43,40.9535
4,Sunhouse,36,39.2500
5,LocknLock,73,36.0822
6,US ARMY,24,35.4167
7,Bear,33,35.0606
8,Kangaroo,34,32.0588
9,BG,74,32.0405


In [ ]:
import numpy as np
import pandas as pd
import math

# 1️. Tính số lượng sản phẩm thuộc top 10%
top_n = math.ceil(len(products) * 0.1)

# 2️. Sort theo quantity_sold giảm dần
products_sorted = products.sort_values(
    "quantity_sold", 
    ascending=False
).copy()

# 3️. Tạo cột mặc định là Non Best Seller
products_sorted["seller_group"] = "Non Best Seller"

# 4️. Gán top 10% đầu tiên thành Best Seller
products_sorted.iloc[:top_n, 
                     products_sorted.columns.get_loc("seller_group")] = "Best Seller"

# 5️. Phân tích giống SQL
best_seller_analysis = (
    products_sorted
    .groupby("seller_group")
    .agg(
        product_count=("seller_group", "count"),
        avg_price=("price", "mean"),
        avg_discount=("discount_rate", "mean"),
        avg_rating=("rating_average", "mean"),
        avg_reviews=("reviews_count", "mean")
    )
    .round({
        "avg_price": 2,
        "avg_discount": 4,
        "avg_rating": 2,
        "avg_reviews": 2
    })
    .reset_index()
)

best_seller_analysis

,seller_group,product_count,avg_price,avg_discount,avg_rating,avg_reviews
0,Best Seller,1220,410470.45,17.9156,4.66,406.59
1,Non Best Seller,10977,1605904.97,7.7780,2.48,8.94


In [12]:
#6. Phân tích sự khác biệt về số lượng sản phẩm đã bán giữa các nhóm tình trạng tồn kho.
inventory_analysis = (
    products
    .groupby("inventory_status")
    .agg(
        product_count=("inventory_status", "count"),
        avg_quantity_sold=("quantity_sold", "mean")
    )
    .round({"avg_quantity_sold": 0})
    .sort_values("avg_quantity_sold", ascending=False)
)

inventory_analysis

,product_count,avg_quantity_sold
inventory_status,,
available,11946,1390.0
out_of_stock,242,102.0
upcoming,9,43.0


In [ ]:
#7. Phân tích ảnh hưởng của tuổi sản phẩm đến hiệu suất bán hàng
def age_bin(x):
    if 0 <= x <= 30:
        return "0-30"
    elif 31 <= x <= 90:
        return "31-90"
    elif 91 <= x <= 180:
        return "91-180"
    else:
        return ">180"

products["age_bin"] = products["day_ago_created"].apply(age_bin) 

age_analysis = (
    products
    .groupby("age_bin")
    .agg(
        product_count=("age_bin", "count"),
        avg_quantity_sold=("quantity_sold", "mean"),
        avg_rating=("rating_average", "mean"),
        avg_reviews=("reviews_count", "mean")
    )
    .round({
        "avg_quantity_sold": 0,
        "avg_rating": 2,
        "avg_reviews": 0
    })
)

age_analysis = age_analysis.reset_index()

age_analysis

,age_bin,product_count,avg_quantity_sold,avg_rating,avg_reviews
0,0-30,134,3940.0,2.92,136.0
1,31-90,423,39.0,0.65,1.0
2,91-180,654,100.0,0.86,3.0
3,>180,10986,1399.0,2.89,52.0


In [17]:
#8. Ảnh hưởng của Authentic đến hiệu suất bán hàng
products["authentic_group"] = np.where(
    products["is_authentic"] == 1,
    "Authentic",
    "Non Authentic"
)

authentic_analysis = (
    products
    .groupby("authentic_group")
    .agg(
        product_count=("authentic_group", "count"),
        avg_price=("price", "mean"),
        avg_quantity_sold=("quantity_sold", "mean"),
        avg_rating=("rating_average", "mean"),
        avg_discount=("discount_rate", "mean")
    )
    .round({
        "avg_price": 2,
        "avg_quantity_sold": 0,
        "avg_rating": 2,
        "avg_discount": 4
    })
)

authentic_analysis = authentic_analysis.reset_index()

authentic_analysis

,authentic_group,product_count,avg_price,avg_quantity_sold,avg_rating,avg_discount
0,Authentic,10404,1656761.72,1244.0,2.81,9.8600
1,Non Authentic,1793,497403.17,2172.0,2.10,2.5951


In [19]:
#9. Phân nhóm theo mức rating
def rating_bin(x):
    if x >= 4.5:
        return "Excellent (>=4.5)"
    elif x >= 4:
        return "Good (4-4.49)"
    elif x >= 3:
        return "Average (3-3.99)"
    else:
        return "Low (<3)"

products["rating_bin"] = products["rating_average"].apply(rating_bin)

rating_analysis = (
    products
    .groupby("rating_bin")
    .agg(
        product_count=("rating_bin", "count"),
        avg_quantity_sold=("quantity_sold", "mean"),
        avg_reviews=("reviews_count", "mean"),
        avg_price=("price", "mean")
    )
    .round({
        "avg_quantity_sold": 0,
        "avg_reviews": 0,
        "avg_price": 2
    })
    .sort_values("avg_quantity_sold", ascending=False)
)

rating_analysis = rating_analysis.reset_index()

rating_analysis

,rating_bin,product_count,avg_quantity_sold,avg_reviews,avg_price
0,Excellent (>=4.5),5895,2089.0,97.0,841725.62
1,Good (4-4.49),847,541.0,25.0,426246.78
2,Average (3-3.99),199,152.0,6.0,427403.70
3,Low (<3),5256,56.0,0.0,2420231.34


In [20]:
#10. Ảnh hưởng của FreeShip Xtra
products["freeship_group"] = np.where(
    products["is_freeship_xtra"] == 1,
    "FreeShip Xtra",
    "No FreeShip"
)

freeship_analysis = (
    products
    .groupby("freeship_group")
    .agg(
        product_count=("freeship_group", "count"),
        avg_price=("price", "mean"),
        avg_quantity_sold=("quantity_sold", "mean"),
        avg_discount=("discount_rate", "mean")
    )
    .round({
        "avg_price": 2,
        "avg_quantity_sold": 0,
        "avg_discount": 4
    })
)

freeship_analysis = freeship_analysis.reset_index()

freeship_analysis

,freeship_group,product_count,avg_price,avg_quantity_sold,avg_discount
0,FreeShip Xtra,9892,1633078.04,573.0,7.1215
1,No FreeShip,2305,856566.07,4254.0,15.9610
